# Computing Minimal Surfaces with Differential Forms

![Minimal surfaces computed in this notebook](assets/gallery.png)

Dip a wire loop into soapy water and the film that forms spans the loop with the
least possible area. Finding that surface for an arbitrary closed curve is
**Plateau's problem**, and it is genuinely hard: the space of surfaces is not
convex, the topology of the answer is not known in advance, and mesh-based
methods tend to tangle themselves when they guess it wrong.

This notebook works through a method that avoids all of that by never
representing the surface as a mesh at all. Following

> Stephanie Wang and Albert Chern,
> [*Computing Minimal Surfaces with Differential Forms*](https://doi.org/10.1145/3450626.3459781),
> ACM Transactions on Graphics 40(4), 2021,

the surface becomes a *differential form* on a periodic box, the area functional
becomes a norm on forms, and the whole problem becomes convex — so a standard
optimizer converges to the global minimum no matter where it starts.

The algorithms live in `src/` and are imported here rather than defined inline,
so the exposition and the tested code cannot drift apart. Working through the
paper turned up several published details that are wrong or unstated; those are
collected in the last section, each with a regression test in `tests/`.

In [ ]:
import numpy as np

from src import curves, extract, spectral
from src.grid import Grid
from src.initial_guess import compute_initial_guess
from src.plateau import solve_plateau

---
## 1. Plateau's problem

Let $M$ be a three-dimensional ambient space, for now a bounded subset
$M \subset \mathbb{R}^3$ with the Euclidean metric. The classical problem is:

> Given a closed curve $\Gamma \subset M$, find an oriented surface
> $\Sigma \subset M$ bordered by $\Gamma$ that minimizes the area functional.

Mathematically,

$$
\min_{\Sigma \, : \, \partial \Sigma = \Gamma} \text{Area}(\Sigma).
$$

### Examples

The circle $\Gamma = \mathbb{S}^1$ has the obvious answer: a flat disc of area
$\pi r^2$. That case is worth keeping in mind throughout, because it is the one
place where we know the exact answer and can therefore *check* the solver rather
than merely admire its output.

A trefoil knot shows why this is hard in general. Its minimal surface is a
genuine Seifert-like spanning surface — nothing disc-shaped, and not something
you would want to have to guess the topology of ahead of time.

![Minimal surfaces for a circle and a trefoil knot](assets/plateau_examples.png)

### Existing approaches

Many approaches have been proposed:

- In 1927 Jesse Douglas gave the first general existence proof, along with a
  finite-difference scheme.
- A mean-curvature-flow approach by Dziuk and Hutchinson (1990).
- An $H^1$ Sobolev gradient flow by Pinkall and Polthier (1993).

These all evolve an explicit surface, which means committing to a topology up
front and hoping it is right. The method here instead rests on *geometric measure
theory*, where surfaces and curves are modelled as **integral currents** — objects
defined by how they integrate against test forms rather than by any
parametrization. Topology stops being something you choose and becomes something
the optimizer discovers.

To get there we need the language of differential forms.

---
## 2. Curves and surfaces as differential forms

The reformulation that makes this problem convex is expressed in vector fields
and differential forms, so this section builds the vocabulary: vector fields on
smooth manifolds, then differential forms, and finally the Dirac-δ forms that let
a curve or a surface be treated as a form in its own right.

### Vector fields

The basic object of study is a vector field. Usually vector fields are first
introduced as simply functions on some euclidean space (2 or 3 -dimensional)
which map an "arrow" to each point in space. Formally, for example a
2-dimensional vector field is a function $f: \mathbb{R}^2 \to \mathbb{R}^2$,
usually with some notion of smoothness included as well to make the function
behave *nicely*.

The basic definition works well in an Euclidean setting and allows us to define
some other operators, such as the divergence and curl operators (in suitable
dimensions). Furthermore, it allows a simple mental picture of what is going on.
One can imagine the speed and direction of wind on the globe as a vector field
defined on the 2-sphere, as the prototypical example.

However, in the more general context of differential geometry and smooth
manifolds, this basic definition does not make sense — if we don't have global
coordinates for our space, but simply a local coordinate chart at each point $p$
on the smooth manifold $M$, then we cannot define this vector field globally for
each point on the surface.

To remedy this, we can modify the definition of a vector field to be more
abstract using things called *derivations*.

#### Vector fields as derivations

At each point $p \in M$, we consider all the smooth real-valued functions on $M$,
denoted $C^\infty(M)$. A tangent vector at $p$ can then be thought of as a
**directional derivative operator** acting on these functions. That is, a vector
$X_p \in T_pM$ is defined as a linear map $X_p : C^\infty(M) \to \mathbb{R}$
which satisfies the **Leibniz rule**:

$$
X_p(fg) = X_p(f)g(p) + f(p)X_p(g)
$$

This is exactly the rule you expect from a derivative operator — it's how the
derivative of a product behaves. Such a map is called a **derivation**. A vector
field $X$ is then the map from a point $p \in M$ to this linear map $X_p$.
Formally, $X: p \mapsto X_p$. And the application of a vector field to a smooth
function produces a map from a point to a real number,
$X(f): p \mapsto \mathbb{R}$.

#### Why is this useful?

This approach gives us a definition of vectors that:

- **Works on any smooth manifold**, whether or not we have coordinates.
- Is **intrinsic**, i.e. does not rely on choosing coordinates or embedding the
  manifold in some higher-dimensional space.
- Naturally leads to the dual concept of differential forms, which are linear
  functionals on vectors.

#### Coordinate basis vectors and derivations

In $\mathbb{R}^n$, we usually express a vector as a linear combination of the
standard basis vectors: $v = a^1 e_1 + a^2 e_2 + \dots + a^n e_n$, where each
$e_i$ points in the direction of the $x^i$-axis. On a smooth manifold $M$, there
is no global coordinate system, but around any point $p \in M$, we can choose a
local coordinate chart $(x^1, \dots, x^n)$. In this chart, we define the
**coordinate vector fields**
$\left( \frac{\partial}{\partial x^1}, \dots, \frac{\partial}{\partial x^n} \right)$
using differentiation:
$\frac{\partial}{\partial x^i} (f)(p) = \frac{\partial f}{\partial x^i}(p)$.
In other words, a coordinate vector field simply differentiates the input
function $f \in C^\infty(M)$ at point $p$ in the "direction" of the specific
coordinate, leaving other coordinates alone — exactly the partial derivative by
definition.

We define the **tangent space** $T_pM$ to be the set of all derivations at $p$.
The collection
$\left( \frac{\partial}{\partial x^1} \right)_p, \dots, \left( \frac{\partial}{\partial x^n} \right)_p$
forms a **basis** of $T_pM$. Therefore, any vector $X_p \in T_pM$ can be uniquely
expressed as some linear combination
$X_p = a^1 \left( \frac{\partial}{\partial x^1} \right)_p + \dots + a^n \left( \frac{\partial}{\partial x^n} \right)_p$
and acts on functions as a directional derivative:
$X_p(f) = a^1 \frac{\partial f}{\partial x^1}(p) + \dots + a^n \frac{\partial f}{\partial x^n}(p)$.
So in local coordinates, a vector at a point $p$ is fully described by how it
differentiates functions along coordinate directions.

#### Example

Let $f(x, y) = x^2 + x y$ and let $p = (1, 2)$. Define a vector field
$X_q = 3 y\frac{\partial}{\partial x} - x\frac{\partial}{\partial y}$, for point
$q = (x, y)$. Therefore the derivation (vector) at $p$, $X_p$, is
$3 \cdot 2 \frac{\partial}{\partial x} - \frac{\partial}{\partial y} = 6 \frac{\partial}{\partial x} - \frac{\partial}{\partial y}$.
We compute the partial derivatives, $\frac{\partial f}{\partial x} = 2x + y$ and
$\frac{\partial f}{\partial y} = x$. At the point $p = (1, 2)$, we get
$\frac{\partial f}{\partial x}(p) = 4$ and $\frac{\partial f}{\partial y}(p) = 1$.
Thus $X_p(f) = 6 \cdot 4 - 1 \cdot 1 = 23$.

Picking another point $q = (-2, 0)$, we obtain another value for the application
of the vector field $X$ to $f$,
$X_q(f) = (3 \cdot 0 \frac{\partial}{\partial x} + 2 \frac{\partial}{\partial y})(f)(q) = -4$.

#### Summary

Vectors on manifolds are best understood as **derivations** — operators acting on
smooth functions, satisfying the Leibniz rule. The coordinate derivations
$\frac{\partial}{\partial x^i}$ provide a natural basis for $T_pM$, and every
vector can be written as a combination of them. This formalism is local,
intrinsic, and independent of any embedding into Euclidean space.

This view becomes powerful when combined with differential forms, because it
provides the foundation for defining integration, exterior differentiation, and
the general machinery of geometric analysis on manifolds.

### Differential forms

Differential forms are the next level up from vector fields: a differential form
is a linear functional on the space of vector fields. In other words, a
differential 1-form takes a vector field $X$ and produces a real number for each
point on the manifold $M$. Likewise, a differential $k$-form takes $k$ vector
fields and produces a real number for every point on the manifold. A differential
0-form is defined to be just a scalar function, that is a function from the
manifold $M$ to the real numbers. Intuitively, a differential form measures how
well the vector field aligns with the form at some point on the manifold. A good
way of visualizing differential forms in low dimensions can be found for example
[here by Dan Piponi](http://yaroslavvb.com/papers/notes/piponi-on.pdf) (1998).

#### Differential form basis

A differential 1-form can look for example something like this,
$\omega = 2\,dx - 5x\,dy$. The quantities $dx$ and $dy$ are the dual 1-forms of
the coordinate directions
$\frac{\partial}{\partial x}, \frac{\partial}{\partial y}$; they are defined such
that the application of the basis form to the corresponding coordinate basis
vector produces simply $1$ for all points $p \in M$. This hints at the fact that,
like vector fields, 1-forms can be formed by a simple linear combination of those
basis 1-forms.

Since there can however be differential forms with higher degrees, for example
$2$-forms, we can use the wedge product ($\wedge$) to combine lower degree forms
to build higher ones. For example we can have a 2-form on a 3-dimensional
manifold, like $\eta = 2\, dx \wedge dy - 7x\, dy \wedge dz + xy\, dx \wedge dz$.
The details of the wedge product are left out here for brevity, but can be found
in any standard differential geometry text.

#### A 1-form example

Take the 1-form $\omega = 2\,dx - 5x\,dy$, the point $p = (1, 3)$, and the vector
field $X = y \frac{\partial}{\partial x} + x \frac{\partial}{\partial y}$. At
point $p$, the specific vector is
$X_p = 3 \frac{\partial}{\partial x} + 1 \frac{\partial}{\partial y}$. Applying
the 1-form to this vector gives
$\omega(X_p) = 2 \cdot 3 - 5 \cdot 1 \cdot 1 = 6 - 5 = 1$. So at the point
$p = (1, 3)$, the 1-form $\omega$ evaluates to $1$ when applied to the vector
field $X$.

If we drop the specific point and instead apply $\omega$ to the whole vector
field, we get a scalar *function* on the manifold rather than a single number:
$\omega(X) = 2y - 5x^2$, which can be evaluated anywhere — at $p = (1,3)$ it gives
$6 - 5 = 1$ as before.

This gives an idea of how 1-forms act like "detectors" or "probes" that measure
the components of vector fields, weighted by their coefficients and position on
the manifold.

#### A 2-form example

Let us now consider a differential 2-form, which takes in two vectors and
produces a real number. A typical example in $\mathbb{R}^3$ might look like
$\eta = x\,dy \wedge dz + y\,dz \wedge dx + z\,dx \wedge dy$.

Evaluated on a pair of vectors
$v = \frac{\partial}{\partial y} + \frac{\partial}{\partial z}$,
$w = \frac{\partial}{\partial x} + 2\frac{\partial}{\partial z}$, the value of the
2-form at point $p = (1, 2, 3)$ is computed using the wedge products.

$dy \wedge dz (v, w)$:
$$
dy \wedge dz (v, w) = \det \begin{bmatrix} v^y & v^z \\ w^y & w^z \end{bmatrix} = \det \begin{bmatrix} 1 & 1 \\ 0 & 2 \end{bmatrix} = 2
$$

$dz \wedge dx (v, w)$:
$$
\det \begin{bmatrix} v^z & v^x \\ w^z & w^x \end{bmatrix} = \det \begin{bmatrix} 1 & 0 \\ 2 & 1 \end{bmatrix} = 1
$$

$dx \wedge dy (v, w)$:
$$
\det \begin{bmatrix} v^x & v^y \\ w^x & w^y \end{bmatrix} = \det \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} = -1
$$

Now we evaluate $\eta(v, w)$ at $p = (1, 2, 3)$:

- The first term gives $x \cdot dy \wedge dz (v, w) = 1 \cdot 2 = 2$
- The second term gives $y \cdot dz \wedge dx (v, w) = 2 \cdot 1 = 2$
- The third term gives $z \cdot dx \wedge dy (v, w) = 3 \cdot (-1) = -3$

Putting it together:

$$
\eta(v, w) = 2 + 2 - 3 = 1
$$

So the 2-form $\eta$ evaluates to 1 on the vectors $v$ and $w$ at the point
$(1, 2, 3)$.

### Dirac-δ forms

Having introduced differential forms and their role as intrinsic, coordinate-free
tools for probing vector fields and oriented geometry, we are ready for the class
of generalized forms that makes the whole reformulation work: **Dirac-δ
differential forms**. These are not smooth forms in the usual sense, but rather
distributions — or more precisely **currents**, continuous linear functionals on
the space of smooth compactly supported differential forms. They let us represent
highly singular geometric objects, such as curves or surfaces, as objects that
can still be integrated against smooth test forms.

The key idea is to represent a surface $\Sigma \subset M$ not by an explicit
parametrization or embedding, but by the **current** $\delta_\Sigma$, which
satisfies

$$
\int_M \omega \wedge \delta_\Sigma = \int_\Sigma \omega
$$

for any smooth test form $\omega$ on the ambient manifold $M$. This object behaves
like a "generalized differential form" that is zero almost everywhere, but
supported entirely on the surface $\Sigma$. It captures the geometry of $\Sigma$
in the weak sense — via its action on test forms — making it ideal for variational
formulations where smoothness may not be guaranteed.

In three dimensions the degrees work out so that a **curve** is dual to a
**2-form** $\delta_\Gamma$, and a **surface** is dual to a **1-form**
$\delta_\Sigma$. Intuitively we can visualize these as "impulses". Just as
ordinary differential forms can be visualized as vector fields in three dimensions
via the musical isomorphisms $\sharp$ and $\flat$, Dirac-δ forms can be visualized
the same way, but as fields that are *local* — the vector field vanishes as we
stray away from the associated curve or surface.

Below, both objects are drawn as vector fields coloured by magnitude, with the
boundary curve in red. On the left, $\delta_\Gamma$ hugs the circle and points
along it. On the right, $\delta_\Sigma$ for the corresponding minimal surface sits
on the disc and points along its normal — the form encodes the surface through its
normals.

![Dirac-delta forms for a curve and a surface](assets/dirac_delta_forms.png)

---
## 3. From geometry to convex optimization

Now we put that machinery to work and turn Plateau's problem into a convex
optimization problem. This convex formulation bypasses meshes and parametric
surface representations entirely. It works directly in the ambient space, supports
topological flexibility, is robust under discretization, and — most importantly —
guarantees convergence to the global minimum.

### 3.1 Representing the surface

Recall the original statement: given a closed curve $\Gamma \subset \mathbb{R}^3$,
find a surface $\Sigma$ with $\partial \Sigma = \Gamma$ minimizing area. This is
difficult because

- the space of surfaces is non-convex, so there is no clear descent direction;
- topology matters, and different surfaces with the same boundary can have
  different numbers of holes;
- mesh-based methods are unstable if the topology is guessed incorrectly.

These difficulties are overcome by treating the surface not as a strict geometric
object but as a Dirac-δ current. We represent a surface
$\Sigma \subset \mathbb{R}^3$ by a 1-form $\delta_\Sigma \in \Omega^1(M)$ and the
boundary curve $\Gamma$ by a 2-form $\delta_\Gamma \in \Omega^2(M)$, where
$\Omega^k(M)$ denotes differential $k$-forms on $M$. They satisfy

$$
\int_M \omega \wedge \delta_\Sigma = \int_\Sigma \omega,
\qquad
\int_M \eta \wedge \delta_\Gamma = \int_\Gamma \eta ,
$$

so the Dirac-δ forms characterize the geometric objects in a weak sense, much like
weak solutions of partial differential equations.

The area functional then becomes a norm. For a 1-form $\eta$, define the **mass
norm**

$$
\| \eta \|_{\text{mass}} = \sup_{\omega \in \Omega^2(M), \|\omega\|_{\max} \leq 1} \int_M \omega \wedge \eta .
$$

For $\eta = \delta_\Sigma$ this is exactly the area of $\Sigma$. So the original
problem becomes: find the surface representation with minimum
$\| \delta_\Sigma \|_{\text{mass}}$.

### 3.2 Rewriting the boundary condition

To enforce $\partial \Sigma = \Gamma$, take a test 1-form $\eta$ and apply Stokes'
theorem:

$$
\int_\Gamma \eta = \int_\Sigma d\eta = \int_M d\eta \wedge \delta_\Sigma = \int_M \eta \wedge d\delta_\Sigma
\quad \Longrightarrow \quad d\delta_\Sigma = \delta_\Gamma .
$$

The boundary constraint becomes simply $d \delta_\Sigma = \delta_\Gamma$. This is
the crucial step: the exterior derivative $d$ is a **linear** operator, so what was
a topological side condition is now a linear differential constraint.

### 3.3 Relaxing the problem

So far we are minimizing $\| \delta_\Sigma \|_{\text{mass}}$ subject to
$d\delta_\Sigma = \delta_\Gamma$, but only over $\delta_\Sigma$ that come from
actual surfaces. The natural relaxation is to optimize over *all* 1-forms
satisfying the boundary constraint. The paper shows this relaxation is exact — it
attains the same value — while giving a much larger and better-behaved search
space:

$$
\boxed{
\min_{\eta \in \Omega^1(M), \ d\eta = \delta_\Gamma} \| \eta \|_{\text{mass}}
}
$$

This is convex with a linear constraint, and appears as **Problem 3** in the paper.

### 3.4 Periodic domains and cohomology

To make the PDE solves fast we work in a periodic box $M = \mathbb{T}^3$ (a
3-torus), which lets us use the FFT. That introduces a new ambiguity:

- surfaces that wrap around the domain may satisfy $d\eta = \delta_\Gamma$ without
  corresponding to any surface in $\mathbb{R}^3$;
- the solution space splits into multiple cohomology classes, each a different
  homology type of surface.

To pin down the right one we compute the **projected area vector**

$$
A = \int_\Sigma N_\Sigma \, dS = \frac{1}{2} \oint_\Gamma \gamma \times d\gamma
$$

and impose the additional constraints

$$
\int_M \vartheta_i \wedge \star \eta = A_i, \quad i = 1,2,3,
$$

where $\vartheta_i = dx_i$ and each $A_i$ is the signed projected area onto the
$yz$, $zx$ and $xy$ planes. These ensure the solution corresponds to a surface
embeddable in $\mathbb{R}^3$ rather than one wrapping around $\mathbb{T}^3$; see
Section 2.6 and Appendix B of the paper for the full argument.

Note that $A$ depends only on the boundary curve, so it can be computed up front.

### 3.5 Splitting off an initial guess

The Helmholtz–Hodge decomposition simplifies things further. The admissible set —
all $\eta$ satisfying both the boundary and cohomology constraints — is an affine
space: it equals $\eta_0 + \operatorname{im}(d)$ for any single feasible $\eta_0$.
Writing $\eta = \eta_0 + d\phi$ for a scalar potential $\phi$ turns the constrained
problem into an unconstrained one:

$$
\boxed{
\begin{aligned}
&\min_{\phi \in C^\infty(M)} \| \eta_0 + d\phi \|_{\text{mass}} \\
&\text{where } \eta_0 \in \Omega^1(M) \text{ satisfies:} \\
&\quad d\eta_0 = \delta_\Gamma \\
&\quad \int_M dx_i \wedge \star \eta_0 = A_i, \quad i = 1,2,3
\end{aligned}
}
$$

This is Problem 6 in the paper, and it is what we actually solve. We have turned a
nonlinear, topology-sensitive geometric variational problem into one that

- is a convex optimization over 1-forms,
- has linear PDE constraints,
- is efficiently solvable by FFT on the periodic domain $\mathbb{T}^3$.

Two things remain: constructing a feasible $\eta_0$, and minimizing over $\phi$.
Before either, we need a discretization.

---
## 4. Discretization

We discretize $\mathbb{T}^3 = [0,1)^3$ by a regular $N^3$ grid of spacing
$h = 1/N$. Everything is collocated at grid vertices and stored as a pointwise
*density* — a 0- or 3-form is an $(N,N,N)$ array, a 1- or 2-form is $(N,N,N,3)$ —
using the periodic identification of vertices, edges and faces the paper describes
in its Section 3.

Collocation is not a stylistic choice. The mass norm's proximal operator (§6)
thresholds on $|X_v| = \sqrt{\sum_i X_{v,i}^2}$ at a *single point*, so the three
components have to live at the same place for that step to stay closed-form.

### Choosing the exterior derivative

Within collocation there is still a choice of difference stencil, and it matters
far more than it looks. We take $d$ to be the **forward** difference

$$(d\phi)_i(v) = \frac{\phi(v + h e_i) - \phi(v)}{h},
\qquad \widehat{d}_i = \frac{e^{i k_i h} - 1}{h},$$

with the codifferential its exact adjoint. Two properties follow, and we need
both.

**It pairs with the Poisson solver.** Summing $|\widehat{d}_i|^2$ gives
$4\sum_i \sin^2(k_i h/2)/h^2$ — *exactly* the 7-point stencil the paper's
Algorithm 3 inverts. The $\phi$-step is an argmin whose normal equations are
$d^\top d\,\phi = d^\top(\cdot)$, so it is only solved exactly when the Poisson
kernel really is $d^\top d$ for the same $d$ used to form $d\phi$ afterwards. The
paper defines $D$ as the midpoint rule (its eq. 23) while inverting the 7-point
stencil; those are different operators, and composing them leaves a **71%
relative residual** in a step that is supposed to be exact. That alone voids the
convergence guarantee. Forward differences restore the pairing — the paper's
Poisson solver was right all along.

**It localizes.** The gradient of a step occupies a single cell. This matters
because the minimizer is a *discontinuous* object, a Dirac-δ form concentrated on
a surface, and the mass norm sums $|X|$ without letting oscillations cancel. A
spectral derivative is exact and rings across the entire domain, charging
$L_1 = 7.18$ for a jump that forward differences localize to $2.0$ — and it
converges to a mass about 62% above the true area. Exactness buys nothing if the
discretization cannot represent the answer.

![Comparison of difference stencils](assets/operators.png)

The left panel shows the three candidates differentiating a step; the right shows
their $d^\top d$ symbols against the paper's Poisson kernel. The forward
difference (green) lies exactly under the dashed kernel. The midpoint rule
(orange) not only misses it but vanishes at the Nyquist frequency, putting the
checkerboard mode in its kernel — which is presumably why the paper reached for
the 7-point stencil in the first place.

Diagonalizing by FFT keeps the inverse Laplacian exact while the operators
themselves stay local.

In [ ]:
# The discrete exterior calculus is exact, not merely consistent.
grid = Grid(32)
rng = np.random.default_rng(0)
phi = rng.standard_normal(grid.res)
eta = rng.standard_normal((*grid.res, 3))

print(f"d(d(phi)) = 0                     : {np.abs(spectral.d1(grid, spectral.d0(grid, phi))).max():.1e}")
print(f"<d0 phi, eta> - <phi, delta1 eta> : "
      f"{float((spectral.d0(grid, phi)*eta).sum()) - float((phi*spectral.delta1(grid, eta)).sum()):.1e}")

lap = spectral.delta1(grid, spectral.d0(grid, phi))
print(f"delta1(d0(phi)) == Laplacian      : {np.abs(lap - spectral.laplace_psd(grid, phi)).max()/np.abs(lap).max():.1e}")

seven_point = sum(4/grid.h**2 * np.sin(k*grid.h/2)**2 for k in grid.k_space)
print(f"d^T d == paper's Algorithm 3      : {np.abs(grid.laplace_symbol - seven_point).max()/seven_point.max():.1e}")

# The phi-step is therefore an exact projection (this residual was 0.71 before).
Y = rng.standard_normal((*grid.res, 3))
resid = spectral.delta1(grid, spectral.d0(grid, spectral.solve_phi(grid, Y)) - Y)
print(f"phi-step residual                 : {np.abs(resid).max()/np.abs(spectral.delta1(grid, Y)).max():.1e}")

---
## 5. The initial guess $\eta_0$

We need one 1-form satisfying both constraints: the boundary condition
$d\eta_0 = \delta_\Gamma$ and the cohomology conditions
$\int_M dx_i \wedge \star \eta_0 = A_i$. It does not have to be a good surface —
just a feasible one. The optimizer takes it from there.

**Discretizing $\delta_\Gamma$.** We deposit each curve segment's tangent vector
onto the eight surrounding vertices with trilinear weights, giving the current
density $J(x) = \oint_\Gamma \delta^3(x - \gamma)\, d\gamma$. Its flux through any
surface is that surface's signed intersection number with $\Gamma$, which is the
defining property. (The paper scans every grid face against every curve segment;
deposition computes the same object in time linear in the number of segments
rather than $O(N^3 M)$.)

**Biot–Savart.** With $\delta_\Gamma$ in hand, solve
$\Delta \psi = \delta_\Gamma$ componentwise and set $\eta_0 = \delta \psi$. Since
the Hodge Laplacian on 2-forms splits as $\Delta = d\delta + \delta d$, and
$d\delta_\Gamma = 0$ for a closed curve, this gives $d\eta_0 = \delta_\Gamma$
exactly. The name is not an analogy: this is literally the magnetic field of a
current loop, and the middle panel below is the familiar dipole pattern.

This step needs the **positive** semi-definite Laplacian. Solving with the
analyst's sign gives $d\eta_0 = -\delta_\Gamma$ — the spanning surface with
reversed orientation, which then fights the $+A$ cohomology correction.

**Cohomology.** $\delta\psi$ has zero mean by construction, so in density units on
the unit cube the correction is just adding the constant $A$, giving
$\text{mean}(\eta_0) = A$.

![The initial guess](assets/initial_guess.png)

The slice is at $y = 0.5$, so the circle appears as the two red points where it
pierces the plane. Note the third panel: $\eta_0$ is smeared across the whole box.
It is feasible, but its mass is roughly twice the true area — there is real work
for the optimizer to do.

Seen in three dimensions as flowing field lines, the resemblance to a magnet is
exact — this *is* the magnetic field of a current loop:

<video src="assets/anim_initial_guess.mp4" controls loop muted playsinline width="640"></video>

In [ ]:
grid = Grid(48)
radius = 0.3
gamma = lambda t: curves.circle(t, radius=radius)
exact = np.pi * radius**2

guess = compute_initial_guess(grid, gamma)

print(f"d(eta_0) = delta_Gamma, relative residual : {guess.residual:.2e}")
print(f"area vector A                             : {np.round(guess.area, 5)}  (exact z: {exact:.5f})")
print(f"mean(eta_0)                               : {np.round(guess.eta_0.reshape(-1, 3).mean(axis=0), 5)}")
print(f"mass(eta_0)                               : {spectral.mass_norm(grid, guess.eta_0):.5f}"
      f"   vs true minimum {exact:.5f}")

---
## 6. Solving: ADMM

We minimize $\|\eta_0 + d\phi\|_{\text{mass}}$ by splitting it. Introduce $X$,
constrained to equal $d\phi + \eta_0$, and alternate:

1. **$\phi$-step** — a Poisson solve, projecting onto the constraint;
2. **$X$-step** — a pointwise shrinkage, the proximal operator of the mass norm,
   $X_v = \max\!\left(1 - \tfrac{1}{\tau |Z_v|},\, 0\right) Z_v$;
3. **dual update** — $\lambda \mathrel{+}= \tau (d\phi - X + \eta_0)$,

with Nesterov acceleration and restart (Goldstein et al. 2014). Two details
deserve attention.

**Where $\lambda$ enters the shrinkage.** Completing the square on the paper's own
eq. (28), $|X| - \langle\lambda,X\rangle + \tfrac{\tau}{2}|X - W|^2$, gives
$Z = W + \lambda/\tau$. Its eq. (29) prints
$Z \leftarrow \tau\lambda + d\phi + \eta_0$, contradicting its own Algorithm 2,
which uses $\lambda/\tau$. The two agree only at $\tau = 1$ — the default — so the
error stays invisible until someone tunes $\tau$.

**Choosing $\tau$.** The paper gives no guidance at all, and the parameter cannot
be scale-free: the shrinkage threshold is $1/\tau$ in absolute field units, while
$|\eta_0|$ is set by the curve and the resolution. At $\tau = 1$ with a typical
$\eta_0$ whose bulk magnitude is well under 1, the very first $X$-step thresholds
the entire field to zero. Here $\tau$ defaults to a value derived from the mean
mass density of $\eta_0$, which makes it resolution-independent. It affects the
convergence *rate*, not the answer — a useful consistency check, and the
regression test for the erratum above.

In [ ]:
# rtol=1e-3 is plenty for an area estimate; see the note below on why.
solution = solve_plateau(gamma, resolution=48, rtol=1e-3, max_iter=600)
initial_mass = spectral.mass_norm(solution.grid, solution.eta_0)

print(solution)
print(f"mass  = {solution.mass:.5f}   exact = {exact:.5f}   error = {(solution.mass-exact)/exact:+.2%}")
print(f"mass(eta_0) was {initial_mass:.5f}, so the optimizer removed "
      f"{1 - solution.mass/initial_mass:.0%} of it")

![ADMM convergence](assets/admm_convergence.png)

The mass starts *below* the true area and climbs. That is not a bug: $\lambda$
begins at zero, so the first shrinkage sees no dual correction and thresholds most
of the field away. It recovers over the following iterations.

The right panel shows why the stopping rule was changed. The paper's criterion $c$
measures only how far the iterates moved, so it falls whether the method converges
or merely stalls. We stop instead on the two KKT conditions — primal feasibility
$d\phi - X + \eta_0 \to 0$ and dual feasibility $d^\top\lambda \to 0$ ($\phi$
carries no cost, so that *is* the dual condition). Residuals decay roughly like
$1/k$, but notice that the mass is worth four digits an order of magnitude
earlier, which is why a loose `rtol` suffices for area estimates.

Watching $|\eta|$ on a slice through the disc makes the mechanism clear: the
diffuse Biot–Savart field collapses onto the surface as the shrinkage
progressively kills everything not needed to satisfy the constraint.

![The field collapsing onto the surface](assets/admm_iterations.png)

In three dimensions the same thing is a solid shrinking onto a sheet. Below is an
isosurface of $|\eta|$, taken at 30% of its current maximum, across the
iterations: it begins as a fat diffuse lens and is squeezed onto the disc, while
the peak magnitude climbs from 3.2 to 14.7 as the mass concentrates.

<video src="assets/anim_optimization.mp4" controls loop muted playsinline width="640"></video>

Note what is *not* animated here. The extracted mesh would show nothing: the
boundary constraint holds from iteration 1, so the level set already encodes a
disc the whole way through. What actually evolves is the support of $\eta$.

---
## 7. From $\eta$ to a mesh

The optimizer's output is an impulse concentrated on the surface, not a mesh. To
render it we find the 0-form $u$ whose differential best matches $\eta$,

$$u = \arg\min_u \|du - \delta_\Sigma\|_{L^2},$$

which is the same normal-equation solve as the $\phi$-step. The theory (paper
eq. 40–41) says $u$ then has a jump of exactly 1 across $\Sigma$ and is smooth
elsewhere, so any isosurface taken inside that jump contains the surface.

An isosurface is always closed, so it necessarily continues past the intended
boundary $\Gamma$. The excess is clipped using $|\eta|$, which vanishes off the
surface — that is what turns a closed level set into a surface *with boundary*.

![Level set extraction](assets/surface_extraction.png)

In [ ]:
vertices, faces = extract.extract_surface(solution)

print(f"{len(vertices)} vertices, {len(faces)} triangles")
print(f"mesh area   = {extract.surface_area(vertices, faces):.5f}")
print(f"mass norm   = {solution.mass:.5f}")
print(f"exact       = {exact:.5f}")

# The mesh should reach the boundary curve without overshooting it.
radial = np.hypot(vertices[:, 0] - 0.5, vertices[:, 1] - 0.5)
print(f"max mesh radius = {radial.max():.4f}  (curve radius {radius})")

---
## 8. Results

Nothing about the solver is specialized to any of these. A boundary is any
callable $t \mapsto \mathbb{R}^3$ on $[0,1)$, an array of polyline points, or a
*list* of either for a multi-component boundary — the Borromean rings below are
three separate closed loops, and their spanning surface is genuinely
three-dimensional because the rings, though pairwise unlinked, cannot be separated
as a triple.

![Gallery of minimal surfaces](assets/gallery.png)

<video src="assets/anim_gallery.mp4" controls loop muted playsinline width="640"></video>

```python
solution = solve_plateau(curves.trefoil, resolution=64)
vertices, faces = extract.extract_surface(solution)

# multi-component boundaries work the same way
solution = solve_plateau(curves.borromean_rings(), resolution=64)
```

---
## 9. Validation

Any *planar* curve has a known answer — its minimal surface is simply the region it
bounds — which gives closed-form values to check against, rather than checking the
solver against itself. A circle gives $\pi r^2$, an ellipse $\pi a b$, and a
polygon its shoelace area.

![Convergence under refinement](assets/refinement.png)

The error is first order in $h$, and its magnitude has a clean explanation: the
discrete surface is thickened by about half a cell along its rim. The dashed line
is $\tfrac{h}{2} \times \text{perimeter}$, with no fitted constants. Mollifying
$\delta_\Gamma$ over one cell (the default) introduces a second error term of
opposite sign that partly cancels the first, lowering the absolute error at every
resolution tested at the cost of a flatter convergence curve; pass
`sigma_cells=0.0` to measure pure discretization error.

The cell below reproduces the check across all three shapes. The last column is
that same half-cell boundary layer measured on each — the triangle's larger
*relative* error is entirely its higher perimeter-to-area ratio, so corners are
handled fine.

In [ ]:
cases = [
    ("circle r=0.3",   gamma,                                      np.pi*radius**2, 2*np.pi*radius),
    ("ellipse .35x.2", lambda t: curves.ellipse(t, a=0.35, b=0.2), np.pi*0.35*0.2,  1.760),
    ("triangle",       curves.triangle,                            0.075,           1.2534),
]

print(f"{'curve':>16} {'N':>4} {'mass':>9} {'exact':>9} {'rel err':>9} {'excess/perim/(h/2)':>20}")
for name, g, ex, perimeter in cases:
    for N in (24, 32):
        s = solve_plateau(g, resolution=N, rtol=1e-3, max_iter=400)
        print(f"{name:>16} {N:4d} {s.mass:9.5f} {ex:9.5f} {(s.mass-ex)/ex:+9.2%}"
              f" {(s.mass-ex)/perimeter/(0.5/N):20.2f}")

---
## 10. Corrections to the published algorithm

Several details in the published algorithm are either erroneous or unstated. Each
is fixed in `src/` with a test guarding it; `README.md` carries the full
derivations.

| # | Issue | Consequence |
|---|-------|-------------|
| 1 | The $\phi$-step's Laplacian does not match its own $D$ (eq. 23 vs Algorithm 3) | 71% residual in an exact argmin; no convergence guarantee |
| 2 | eq. (29) uses $\tau\lambda$ where the derivation gives $\lambda/\tau$ | Masked at the default $\tau=1$; breaks on tuning |
| 3 | Algorithm 5 line 21 writes the sharp operator as a difference, but eq. (23) defines it as an average | $X_0$ becomes a derivative of $\eta_0$ instead of $\eta_0$ |
| 4 | The Biot–Savart solve needs the positive semi-definite Laplacian | Otherwise $d\eta_0 = -\delta_\Gamma$, reversed orientation |
| 5 | Algorithm 5 lines 12–18 drop the $\lvert V \rvert$ normalization | Cohomology correction off by $N^3$ |
| 6 | No guidance on $\tau$, which cannot be scale-free | At $\tau=1$ the first $X$-step can zero the whole field |
| 7 | The convergence criterion cannot distinguish converged from stalled | Reports success on a stalled run |

Item 1 is the one that actually mattered. It is also the one that took a wrong
turn to find: the first attempt here used a fully *spectral* $d$, chosen because it
makes $d^\top d$ exact — and it converged to a mass 62% too high, because the
minimizer is discontinuous and spectral operators ring. Locality turned out to
matter as much as exactness, and forward differences are the only choice with
both.

---

## References

1. Stephanie Wang and Albert Chern. 2021. Computing Minimal Surfaces with
   Differential Forms. *ACM Transactions on Graphics* 40, 4, Article 113.
   [doi:10.1145/3450626.3459781](https://doi.org/10.1145/3450626.3459781)
2. Tom Goldstein, Brendan O'Donoghue, Simon Setzer, Richard Baraniuk. 2014. Fast
   Alternating Direction Optimization Methods. *SIAM Journal on Imaging Sciences*
   7, 3.
3. Jesse Douglas. 1931. Solution of the Problem of Plateau. *Transactions of the
   AMS* 33, 1.
4. Ulrich Pinkall and Konrad Polthier. 1993. Computing Discrete Minimal Surfaces
   and Their Conjugates. *Experimental Mathematics* 2, 1.
5. Michael Kazhdan, Matthew Bolitho, Hugues Hoppe. 2006. Poisson Surface
   Reconstruction. *SGP '06*.